# ForgeIR Milestone 12: real NVIDIA CUDA validation

## Goal

Build the optional handwritten CUDA backend on a Colab NVIDIA GPU, compare every supported kernel with PyTorch CUDA, record CUDA-event timings and hardware metadata, and export the actual result JSON. A failed command raises immediately; this notebook never converts a failed run into a success.

## Setup

Select a GPU runtime in Colab before running. Set `REPOSITORY_URL` to the real ForgeIR Git URL. The notebook creates a clean clone and a virtual environment that can see Colab's preinstalled PyTorch CUDA package.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "SET_ME_TO_THE_FORGEIR_GIT_URL"
REPOSITORY_REVISION = "main"
CLONE_DIRECTORY = Path("/content/ForgeIR")

if REPOSITORY_URL.startswith("SET_ME"):
    raise RuntimeError("Set REPOSITORY_URL before running the notebook")
if CLONE_DIRECTORY.exists():
    raise RuntimeError(f"Refusing to reuse stale clone: {CLONE_DIRECTORY}")
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_REVISION, REPOSITORY_URL, str(CLONE_DIRECTORY)], check=True)
os.chdir(CLONE_DIRECTORY)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", ".venv-cuda"], check=True)
venv_python = CLONE_DIRECTORY / ".venv-cuda" / "bin" / "python"
venv_pip = CLONE_DIRECTORY / ".venv-cuda" / "bin" / "pip"
subprocess.run([str(venv_pip), "install", "--requirement", "backends/cuda/requirements-colab.txt"], check=True)
subprocess.run([str(venv_python), "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], check=True)


## Steps

Configure and build the CUDA Release preset. The environment is explicit so CMake discovers pybind11 from this virtual environment rather than a user-specific path.

In [ ]:
cuda_environment = os.environ.copy()
cuda_environment["VIRTUAL_ENV"] = str(CLONE_DIRECTORY / ".venv-cuda")
cuda_environment["PATH"] = str(CLONE_DIRECTORY / ".venv-cuda" / "bin") + os.pathsep + cuda_environment["PATH"]
subprocess.run(["bash", "scripts/linux/configure_cuda.sh"], check=True, env=cuda_environment)
subprocess.run(["bash", "scripts/linux/build_cuda.sh"], check=True, env=cuda_environment)


## Checks

Run all C++/CLI tests, the PyTorch CUDA parity matrix, and the documented profiling protocol. The validation command writes its JSON only after all cases pass.

In [ ]:
subprocess.run(["bash", "scripts/linux/test_cuda.sh"], check=True, env=cuda_environment)
result_path = CLONE_DIRECTORY / "benchmarks" / "results" / "cuda" / "milestone_12.json"
if not result_path.is_file():
    raise RuntimeError(f"CUDA validation did not create {result_path}")
result = json.loads(result_path.read_text(encoding="utf-8"))
if result.get("status") != "passed":
    raise RuntimeError(f"CUDA validation status is not passed: {result.get('status')!r}")
if not result.get("hardware", {}).get("gpu_model"):
    raise RuntimeError("CUDA result is missing the GPU model")
print(json.dumps({key: result[key] for key in ("status", "case_count", "maximum_absolute_error", "maximum_relative_error", "hardware")}, indent=2, sort_keys=True))


## Next Steps

Export the measured JSON and the complete CTest log. Milestone 12 may be marked complete only after reviewing these artifacts and recording their exact values in `docs/phase_reports/phase_12.md`.

In [ ]:
export_directory = Path("/content/forgeir_cuda_results")
export_directory.mkdir(parents=True, exist_ok=False)
shutil.copy2(result_path, export_directory / result_path.name)
archive_path = shutil.make_archive(str(export_directory), "zip", root_dir=export_directory)
if not Path(archive_path).is_file():
    raise RuntimeError("Failed to create CUDA result archive")
from google.colab import files
files.download(archive_path)
